## Import

In [ ]:
!pip install rasterio matplotlib

## Rasterio

Rasterio is a Python library for reading, writing, and processing raster geospatial data, such as satellite imagery, digital elevation models (DEM), and other raster datasets.

Rasterio allows access to raster metadata, coordinate reference systems (CRS), individual raster bands, and pixel values, while preserving geospatial information.


In [7]:
import rasterio
import matplotlib.pyplot as plt

## How to open dataset

Raster data is stored on disk in files such as GeoTIFF. TIFF (Tagged Image File Format) is an image format used to store both raster pixel values and the corresponding metadata. 
To work with raster data in Rasterio, we first need to open the file using `rasterio.open()`.

The result is a DatasetReader object, which provides access to:
- raster metadata (CRS, resolution, bounds)
- raster bands
- pixel values

In [10]:
dataset = rasterio.open('data/23-05-2020.tiff')

In [ ]:
dict(
    dataset.profile
)

- **count**  
  Number of bands in the raster dataset. For example, a RGB image has `count = 3`.

- **transform**  
  An affine transformation that maps pixel coordinates (row, column) to real-world spatial coordinates. It defines pixel size, rotation, and origin.

- **nodata**  
  A special value used to represent missing or invalid data. Pixels with this value should be ignored in analysis and calculations.

In [ ]:
dataset.bounds

## How to read bands

Raster datasets often contain multiple bands.
Each band can be accessed separately using the `read()` method or to separate using a tuple (n1, n2, ...)

In [ ]:
band4 = dataset.read(4)
band8 = dataset.read(8)

band48 = dataset.read((4,8))
band48.shape

In [ ]:
band4

In [ ]:
plt.imshow(band4, cmap='gray')

## How to write

The `driver` parameter specifies the file format used to write the raster. 

In this notebook, the `GTiff` driver is used, which corresponds to the GeoTIFF format. However you can use `PNG` for example. In this case an additional georeferencing file will be created.

Rasterio datasets are opened using the `with` statement to ensure that the file is properly closed after use.

In [18]:
with rasterio.open(
        'tmp.tif',
        'w',
        driver='GTiff',
        height=band4.shape[0],
        width=band4.shape[1],
        count=1,
        dtype=band4.dtype,
        crs=dataset.crs,
        transform=dataset.transform,
        nodata=dataset.nodata
    ) as file:

    file.write(band4, 1) # saves the NumPy array as a raster band number 1

Rasterio allows writing new raster files by specifying metadata such as:
- image size (width, height)
- number of bands
- data type
- coordinate reference system

Metadata from an existing dataset is often reused and slightly modified to ensure consistency between input and output rasters.

In [19]:
new_meta = dataset.meta.copy()
new_meta.update({
        'count': 1
    })

with rasterio.open(
        'tmp2.tif',
        'w',
        **new_meta
    ) as file:

    file.write(band4, 1)

## Reproject

In [20]:
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np

In [11]:
dst_crs = 'EPSG:4326'

with rasterio.open('tmp.tif') as src:
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds)
    kwargs = src.meta.copy()
    kwargs.update({
        'crs': dst_crs,
        'transform': transform,
        'width': width,
        'height': height
    })

    with rasterio.open('tmp_reprojected.tif', 'w', **kwargs) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source=src.read(i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest)

## Resample

Resampling changes the spatial resolution of a raster. This can be useful when increasing or decreasing pixel size or when aligning rasters with different resolutions.

In [34]:
import rasterio
from rasterio.enums import Resampling


In [35]:
upscale_factor = 2

with rasterio.open("tmp.tif") as dataset:

    # resample data to target shape
    data = dataset.read(
        out_shape=(
            dataset.count,
            int(dataset.height * upscale_factor),
            int(dataset.width * upscale_factor)
        ),
        resampling=Resampling.bilinear
    )

    # scale image transform
    transform = dataset.transform * dataset.transform.scale(
        (dataset.width / data.shape[-1]),
        (dataset.height / data.shape[-2])
    )

In [ ]:
Resampling._member_names_ # Resampling methods

In [38]:
import matplotlib.pyplot as plt

In [ ]:
plt.imshow(data[0])

## Calculate indexes

In [40]:
src = rasterio.open('data/23-05-2020.tiff')

In [41]:
b4 = src.read(4)
b8 = src.read(8)

NIR = (b8 - b4) / (b4 + b8)

In [ ]:
plt.imshow(NIR)
plt.colorbar()

### Tasks

- Calculate any 5 indexes (https://www.indexdatabase.de/db/is.php?sensor_id=96)
- Save each index as a separate file
- Reproject each file to another CRS (For example, 4326)

In [20]:
# TODO